In [ ]:
# %pip install -U pandas numpy gensim rapidfuzz matplotlib

In [2]:
from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from rapidfuzz import fuzz
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)

# نام فایل‌ها را در صورت نیاز تغییر دهید.
DATA_PATH = None  # مثال: Path("my_reviews.csv")
POSSIBLE_INPUTS = [
    Path("sentiment_ready_dataset.csv"),
    Path("test_data(1).csv"),
    Path("test_data.csv"),
]

if DATA_PATH is None:
    existing_files = [p for p in POSSIBLE_INPUTS if p.exists()]
    if not existing_files:
        raise FileNotFoundError(
            "فایل ورودی پیدا نشد. CSV را کنار نوت‌بوک قرار دهید یا DATA_PATH را تنظیم کنید."
        )
    DATA_PATH = existing_files[0]
else:
    DATA_PATH = Path(DATA_PATH)
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"فایل ورودی وجود ندارد: {DATA_PATH.resolve()}")

OUTPUT_DIR = Path("warranty_analysis_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file: {DATA_PATH.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Input file: /mnt/data/sentiment_ready_dataset.csv
Output directory: /mnt/data/warranty_analysis_output


In [3]:
# خواندن CSV؛ در صورت تفاوت Encoding می‌توان encoding را به cp1252 تغییر داد.
try:
    df = pd.read_csv(DATA_PATH, low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(DATA_PATH, low_memory=False, encoding="cp1252")

# یکسان‌سازی نام ستون‌ها برای تشخیص راحت‌تر
original_columns = df.columns.tolist()
df.columns = [str(c).strip().lower() for c in df.columns]

# پشتیبانی از نام‌های رایج ستون‌های دیتاست خام و نسخه پاک‌سازی‌شده
TEXT_CANDIDATES = ["text_for_model", "review_text_clean", "reviewtext", "review_text", "text"]
SUMMARY_CANDIDATES = ["summary_clean", "summary", "review_title"]
PRODUCT_CANDIDATES = ["asin", "product_id", "productid", "product"]
RATING_CANDIDATES = ["overall", "rating", "star_rating", "stars"]


def first_existing(columns, candidates):
    return next((c for c in candidates if c in columns), None)

text_col = first_existing(df.columns, TEXT_CANDIDATES)
product_col = first_existing(df.columns, PRODUCT_CANDIDATES)
rating_col = first_existing(df.columns, RATING_CANDIDATES)
summary_col = first_existing(df.columns, SUMMARY_CANDIDATES)

if text_col is None:
    raise KeyError(
        "ستون متن نظر پیدا نشد. یکی از ستون‌های reviewText، review_text_clean یا text_for_model لازم است."
    )
if product_col is None:
    raise KeyError("ستون شناسه محصول پیدا نشد؛ برای گزارش به تفکیک کالا به asin یا product_id نیاز است.")

# اگر متن اصلی و خلاصه جدا هستند، هر دو وارد تحلیل می‌شوند.
# اگر text_for_model موجود باشد، فرض می‌کنیم متن ترکیبی از قبل ساخته شده و دوباره summary را اضافه نمی‌کنیم.
df["__review_text"] = df[text_col].fillna("").astype(str)
if text_col != "text_for_model" and summary_col is not None:
    df["__review_text"] = (
        df[summary_col].fillna("").astype(str).str.strip()
        + " " + df["__review_text"].str.strip()
    ).str.strip()

df["__product_id"] = df[product_col].astype("string").str.strip()
df.loc[df["__product_id"].isin(["", "<NA>", "nan", "None"]), "__product_id"] = pd.NA

# ستون‌های توصیفی محصول در صورت وجود
brand_col = first_existing(df.columns, ["brand", "brand_name"])
title_col = first_existing(df.columns, ["product_title", "title", "product_name"])

print(f"Rows: {len(df):,}")
print(f"Text column: {text_col}")
print(f"Product ID column: {product_col}")
print(f"Rating column: {rating_col if rating_col else 'NOT FOUND'}")
print(f"Brand column: {brand_col if brand_col else 'not available'}")
print(f"Product title column: {title_col if title_col else 'not available'}")
print("\nColumns:", df.columns.tolist())

if rating_col is None:
    print("\nWARNING: ستون overall/rating در فایل وجود ندارد. vote امتیاز ستاره‌ای نیست و از آن استفاده نمی‌شود.")
else:
    print("\nRating values preview:")
    display(df[rating_col].value_counts(dropna=False).head(10).to_frame("count"))

Rows: 19,997
Text column: text_for_model
Product ID column: asin
Rating column: NOT FOUND
Brand column: brand
Product title column: product_title

Columns: ['asin', 'brand', 'product_title', 'text_for_model', 'review_text_clean', 'summary_clean', 'review_date', 'verified_purchase', 'helpful_votes', 'style_attributes', '__review_text', '__product_id']



In [4]:
TOKEN_PATTERN = re.compile(r"[a-z]+(?:['-][a-z]+)?")


def tokenize_english(text):
    """تبدیل متن انگلیسی به توکن‌های حروفی کوچک‌شده؛ علائم نگارشی حذف می‌شوند."""
    return TOKEN_PATTERN.findall(str(text).lower())


corpus = [tokenize_english(t) for t in df["__review_text"].fillna("")]
corpus = [tokens for tokens in corpus if tokens]
if not corpus:
    raise ValueError("پس از توکن‌سازی هیچ متن معتبری باقی نماند.")

# Word2Vec از بافت واژه‌ها در همین مجموعه نظرات برای یافتن واژه‌های نزدیک استفاده می‌کند.
# در دیتاست‌های کوچک، نتایج embedding ممکن است نویزی باشد؛ به همین دلیل در مراحل بعدی
# هم واژه‌های دامنه‌ای کنترل‌شده و هم غلط‌های املایی به مجموعه اضافه می‌شوند.

w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=40,
    window=5,
    min_count=2,
    workers=1,
    sg=0,             # CBOW: سریع‌تر برای اجرای محلی
    negative=5,
    epochs=3,
    seed=42,
)

print(f"Vocabulary size: {len(w2v_model.wv):,}")
print(f"Training sentences: {len(corpus):,}")

Vocabulary size: 19,285
Training sentences: 19,997


In [5]:
SEED_WORDS = ["warranty", "guarantee"]
EMBEDDING_TOP_N = 20
MIN_COSINE_SIMILARITY = 0.45

embedding_rows = []
embedding_terms = set()

for seed in SEED_WORDS:
    if seed not in w2v_model.wv:
        print(f"'{seed}' در واژگان Word2Vec نیست؛ برای آن واژه مشابه برداری استخراج نشد.")
        continue

    similar_words = w2v_model.wv.most_similar(seed, topn=EMBEDDING_TOP_N)
    for word, score in similar_words:
        if score >= MIN_COSINE_SIMILARITY:
            embedding_terms.add(word.lower())
            embedding_rows.append({
                "seed_word": seed,
                "similar_word": word.lower(),
                "cosine_similarity": float(score),
            })

embedding_df = pd.DataFrame(
    embedding_rows,
    columns=["seed_word", "similar_word", "cosine_similarity"]
).sort_values("cosine_similarity", ascending=False)

display(embedding_df)
embedding_df.to_csv(OUTPUT_DIR / "warranty_embedding_candidates.csv", index=False, encoding="utf-8-sig")

print("\nتوجه: واژه‌های برداری نامزد هستند؛ شباهت بافتی الزاماً به معنی مترادف دقیق نیست.")

,seed_word,similar_word,cosine_similarity
20,guarantee,authorize,0.854444
21,guarantee,confidence,0.847783
22,guarantee,impact,0.845665
23,guarantee,promise,0.842027
24,guarantee,bothering,0.837453
25,guarantee,circumstances,0.836487
26,guarantee,seek,0.832902
27,guarantee,owner,0.829907
28,guarantee,harm,0.828928
29,guarantee,disappoint,0.828794



توجه: واژه‌های برداری نامزد هستند؛ شباهت بافتی الزاماً به معنی مترادف دقیق نیست.


In [6]:
# واژه‌های مرتبط متداول در حوزه ضمانت. از واژه‌های بیش از حد عمومی مانند plan به‌تنهایی
# استفاده نشده تا تعداد تطبیق‌های اشتباه زیاد نشود.
DOMAIN_TERMS = {
    "warranty", "warranties", "warrantee", "warrantees",
    "guarantee", "guarantees", "guaranteed", "guaranteeing",
    "guaranty", "guaranties",
    "coverage", "covered", "covering",
    "protection", "protected", "protectionplan",
    "insurance", "insured", "insurer",
}

# غلط‌های تایپی پرتکرار و گونه‌های نوشتاری نزدیک به دو واژه اصلی
COMMON_MISSPELLINGS = {
    "warrenty", "warrenties", "waranty", "waranties", "warrantey",
    "warrante", "warrantee", "warrantees", "warrenty's",
    "guarentee", "guarentees", "guarenteed", "guarenteeing",
    "garantee", "garantees", "garanteed", "garanty", "guarrantee",
    "guarranteed", "guaranteee", "guaranteees", "guarantee's",
}

# پیدا کردن غلط‌های احتمالی داخل واژگان همین دیتاست با RapidFuzz.
# آستانه نسبتاً بالا انتخاب شده تا کلمات نامرتبط کمتر وارد مجموعه شوند.
FUZZY_THRESHOLD = 86
vocabulary = set(w2v_model.wv.index_to_key)
fuzzy_terms = set()
fuzzy_rows = []

fuzzy_seeds = ["warranty", "guarantee"]
for token in vocabulary:
    if len(token) < 6:
        continue
    best_seed, best_score = max(
        ((seed, fuzz.ratio(token, seed)) for seed in fuzzy_seeds),
        key=lambda item: item[1],
    )
    if best_score >= FUZZY_THRESHOLD and token not in fuzzy_seeds and token not in {"warrant", "warrants"}:
        fuzzy_terms.add(token)
        fuzzy_rows.append({"candidate": token, "closest_seed": best_seed, "similarity": best_score})

fuzzy_df = pd.DataFrame(fuzzy_rows, columns=["candidate", "closest_seed", "similarity"])
if not fuzzy_df.empty:
    fuzzy_df = fuzzy_df.sort_values("similarity", ascending=False)

display(fuzzy_df.head(50))
fuzzy_df.to_csv(OUTPUT_DIR / "warranty_fuzzy_candidates.csv", index=False, encoding="utf-8-sig")

# Word2Vec نامزدها را پیشنهاد می‌کند، اما در این دیتاست بخشی از پیشنهادها (به‌ویژه برای
# guarantee که فراوانی کمی دارد) مترادف دقیق نیستند. پس از بازبینی جدول بالا، فقط موارد
# مرتبط زیر تأیید شده‌اند. این فهرست را می‌توان با توجه به کاربرد پروژه توسعه داد.
APPROVED_EMBEDDING_TERMS = {
    "rma", "replacement", "repair", "manufacturer"
}

# مجموعه نهایی: واژه‌های اصلی + واژه‌های دامنه‌ای + غلط‌های املایی + نامزدهای برداری تأییدشده
ALL_TERMS = (
    set(SEED_WORDS)
    | DOMAIN_TERMS
    | COMMON_MISSPELLINGS
    | fuzzy_terms
    | APPROVED_EMBEDDING_TERMS
)

# عبارت‌های چندواژه‌ای رایج نیز بررسی می‌شوند؛ تطبیق آن‌ها جداگانه انجام خواهد شد.
PHRASES = {
    "service plan", "extended warranty", "extended guarantee",
    "protection plan", "repair coverage", "limited warranty",
    "manufacturer warranty", "manufacturer guarantee",
    "lifetime warranty", "lifetime guarantee",
}

print("Approved embedding terms:", sorted(APPROVED_EMBEDDING_TERMS))
print(f"Exact word terms: {len(ALL_TERMS)}")
print(f"Phrase patterns: {len(PHRASES)}")
print("Sample terms:", sorted(ALL_TERMS)[:60])

,candidate,closest_seed,similarity
1,guaranteed,guarantee,94.736842
2,guarantees,guarantee,94.736842
0,warrenty,warranty,87.500000


Approved embedding terms: ['manufacturer', 'repair', 'replacement', 'rma']
Exact word terms: 43
Phrase patterns: 10
Sample terms: ['coverage', 'covered', 'covering', 'garantee', 'garanteed', 'garantees', 'garanty', 'guarantee', "guarantee's", 'guaranteed', 'guaranteee', 'guaranteees', 'guaranteeing', 'guarantees', 'guaranties', 'guaranty', 'guarentee', 'guarenteed', 'guarenteeing', 'guarentees', 'guarrantee', 'guarranteed', 'insurance', 'insured', 'insurer', 'manufacturer', 'protected', 'protection', 'protectionplan', 'repair', 'replacement', 'rma', 'waranties', 'waranty', 'warrante', 'warrantee', 'warrantees', 'warrantey', 'warranties', 'warranty', 'warrenties', 'warrenty', "warrenty's"]


In [7]:
# الگوی واژه‌ها با مرز کلمه ساخته می‌شود تا مثلاً واژه warranty به‌صورت زیررشته
# ناخواسته در یک کلمه نامرتبط match نشود.
word_pattern = re.compile(
    r"(?<![a-z])(?:" + "|".join(re.escape(w) for w in sorted(ALL_TERMS, key=len, reverse=True)) + r")(?![a-z])",
    flags=re.IGNORECASE,
)
phrase_pattern = re.compile(
    r"(?<![a-z])(?:" + "|".join(re.escape(p) for p in sorted(PHRASES, key=len, reverse=True)) + r")(?![a-z])",
    flags=re.IGNORECASE,
)


def find_aspect_terms(text):
    text = str(text).lower()
    found = set(m.group(0).lower() for m in word_pattern.finditer(text))
    found.update(m.group(0).lower() for m in phrase_pattern.finditer(text))
    return sorted(found)


df["matched_warranty_terms"] = df["__review_text"].map(find_aspect_terms)
df["mentions_warranty"] = df["matched_warranty_terms"].map(bool)

aspect_reviews = df.loc[df["mentions_warranty"]].copy()
aspect_reviews["matched_warranty_terms"] = aspect_reviews["matched_warranty_terms"].map(
    lambda terms: "; ".join(terms)
)

print(f"Total reviews: {len(df):,}")
print(f"Warranty/guarantee-related reviews: {len(aspect_reviews):,}")
print(f"Share of reviews: {len(aspect_reviews) / max(len(df), 1):.2%}")

display(aspect_reviews[[product_col, "matched_warranty_terms", "__review_text"]].head(10))

Total reviews: 19,997
Warranty/guarantee-related reviews: 2,401
Share of reviews: 12.01%


,asin,matched_warranty_terms,__review_text
3,B00FBJ4KYC,protection; protection plan; warranty,DO NOT BUY!!!. I purchased this 4 year protection plan for a security system I had purchased which ended up being defective and returned...
4,B00P7G82TS,replacement,Nightmare - don't buy. I bought one of these and have regretted it ever since. The GPS receiver worked for a day and a half before crapp...
11,B01AOBOAWS,rma,"Noisy Fans, Poor Tech Support, Defective Hardware, AND Leaks!. Summary of Prestige model: Seemingly good build quality. Design met my ne..."
27,B00FZHWFEW,replacement,Cheap mounting screws. One of the standoff screws snapped right in two as soon as it had tension on it. Now I have to spend almost $20 o...
40,B0002UB2K0,replacement,"Strobe light did not work, and neither did the replacement one.. I bought this strobe light to use with my Midland WR-300 weather radio...."
47,B01BGTG41W,replacement; warranty,"Beware of FAT32 format. My drive came with FAT32 format and I did not pay attention. After a few uses, the drive goes bad and I can't up..."
49,B00YY3XHLS,manufacturer,This disk worked fine for 2 weeks. DO NOT BUY THIS HARD DRIVE! This disk worked fine for 2 weeks.. Now the drives has failed and I am no...
56,B006TA2F1M,replacement,Fails.. I had nothing but compliments on this radio until it up and failed. Less than 90 days later I cannot get a replacement. Amazon w...
67,B00SMBFZ0Y,limited warranty; warranty,"good but too much base. Base is too much as other reviews said. Even with the base turned off completely, it reverberates with certain v..."
73,B01H74WDZG,manufacturer,No customer support. Zilch. Nada. Reflected in my review.. I purchased a set of these earbuds a few months back. They worked well for a ...


In [8]:
# نام‌های ستون خروجی خواناتر
rename_map = {"__product_id": "product_id"}
if brand_col:
    rename_map[brand_col] = "brand"
if title_col:
    rename_map[title_col] = "product_title"

# خروجی نظرات مرتبط
preferred_cols = ["__product_id"]
for col in [brand_col, title_col]:
    if col and col not in preferred_cols:
        preferred_cols.append(col)
for col in ["matched_warranty_terms", "__review_text"]:
    if col not in preferred_cols:
        preferred_cols.append(col)
if rating_col:
    preferred_cols.append(rating_col)
for col in ["review_date", "reviewtime", "verified_purchase", "verified", "helpful_votes", "vote"]:
    if col in aspect_reviews.columns and col not in preferred_cols:
        preferred_cols.append(col)

matched_reviews_export = aspect_reviews[preferred_cols].rename(columns=rename_map).copy()
matched_reviews_export = matched_reviews_export.rename(columns={"__review_text": "review_text"})
matched_reviews_path = OUTPUT_DIR / "warranty_guarantee_matched_reviews.csv"
matched_reviews_export.to_csv(matched_reviews_path, index=False, encoding="utf-8-sig")

# نام/برند محصول را نیز در صورت وجود به کلید گروه‌بندی اضافه می‌کنیم.
group_cols = ["__product_id"]
for col in [brand_col, title_col]:
    if col:
        group_cols.append(col)

if rating_col is not None:
    aspect_reviews["__overall_numeric"] = pd.to_numeric(aspect_reviews[rating_col], errors="coerce")
    # امتیازهای خارج از بازه متداول Amazon (1 تا 5) نامعتبر در نظر گرفته می‌شوند.
    valid_rating_mask = aspect_reviews["__overall_numeric"].between(1, 5)
    aspect_reviews.loc[~valid_rating_mask, "__overall_numeric"] = np.nan

    product_report = (
        aspect_reviews.groupby(group_cols, dropna=False)
        .agg(
            aspect_review_count=("__product_id", "size"),
            rating_count=("__overall_numeric", "count"),
            mean_overall=("__overall_numeric", "mean"),
        )
        .reset_index()
    )
    product_report["calculation_status"] = np.where(
        product_report["rating_count"] > 0,
        "calculated",
        "no_valid_overall_values",
    )
else:
    # مهم: vote تعداد رأی مفید بودن review است و Rating نیست.
    product_report = (
        aspect_reviews.groupby(group_cols, dropna=False)
        .agg(aspect_review_count=("__product_id", "size"))
        .reset_index()
    )
    product_report["rating_count"] = 0
    product_report["mean_overall"] = np.nan
    product_report["calculation_status"] = "not_calculated_no_overall_column"
    print(
        "\nمیانگین امتیاز محاسبه نشد: هیچ‌یک از ستون‌های overall/rating/stars در فایل وجود ندارد.\n"
        "برای محاسبه واقعی، فایل ورودی باید امتیاز ستاره‌ای هر نظر را در ستون overall داشته باشد."
    )

product_report = product_report.rename(columns=rename_map)
product_report = product_report.sort_values(
    ["aspect_review_count", "mean_overall"], ascending=[False, False], na_position="last"
)
product_report_path = OUTPUT_DIR / "warranty_guarantee_product_report.csv"
product_report.to_csv(product_report_path, index=False, encoding="utf-8-sig")

display(product_report.head(30))
print(f"\nSaved: {matched_reviews_path.resolve()}")
print(f"Saved: {product_report_path.resolve()}")


میانگین امتیاز محاسبه نشد: هیچ‌یک از ستون‌های overall/rating/stars در فایل وجود ندارد.
برای محاسبه واقعی، فایل ورودی باید امتیاز ستاره‌ای هر نظر را در ستون overall داشته باشد.


,product_id,brand,product_title,aspect_review_count,rating_count,mean_overall,calculation_status
697,B00HCMZ2AC,SanDisk,"SanDisk Clip Sport 8GB MP3 Player, Blue With LCD Screen and MicroSDHC Card Slot- SDMX24-008G-G46B",7,0,NaN,not_calculated_no_overall_column
1142,B00S9SGNNS,Asus,"ASUS Tri-Band Gigabit (AC3200) WiFi Router (Up to 3167 Mbps) with MU-MIMO to ensure Lag-Free Gaming, AiProtection network security power...",7,0,NaN,not_calculated_no_overall_column
370,B008ABOJKS,Asus,ASUS Dual-band 3x3 AC1750 Wifi 4-port Gigabit Router with speeds up to 1750Mbps &amp; AiRadar to strengthens Wireless Connections via Hi...,6,0,NaN,not_calculated_no_overall_column
385,B008VQ8IKY,Western Digital,WD Red 8TB NAS Hard Disk Drive - 5400 RPM Class SATA 6 Gb/s 128MB Cache 3.5 Inch - WD80EFZX,6,0,NaN,not_calculated_no_overall_column
1015,B00OBRE5UE,Samsung,Samsung 850 EVO 500GB 2.5-Inch SATA III Internal SSD (MZ-75E500B/AM),6,0,NaN,not_calculated_no_overall_column
2060,B01FDPW1NK,Samsung,"Samsung Gear Fit2- Pink, Medium/Large",5,0,NaN,not_calculated_no_overall_column
101,B0012YLTR6,APC,"APC 11-Outlet Surge Protector 3020 Joules with Telephone, DSL and Coaxial Protection, SurgeArrest Performance (P11VT3)",4,0,NaN,not_calculated_no_overall_column
107,B0015DYMVO,Belkin,Belkin BZ103050-TVL Mini Surge Protector with USB Charger - 1 AMP (Retail Package),4,0,NaN,not_calculated_no_overall_column
383,B008VQ2Y4K,SilverStone Technology,"SilverStone Technology 450W SFX Form Factor 80 PLUS GOLD Full Modular Power Supply with +12V single rail, Active PFC (ST45SF-G)",4,0,NaN,not_calculated_no_overall_column
593,B00EM5UFP4,NETGEAR,NETGEAR Smart WiFi Router AC1750 Dual Band Gigabit (R6300v2),4,0,NaN,not_calculated_no_overall_column



Saved: /mnt/data/warranty_analysis_output/warranty_guarantee_matched_reviews.csv
Saved: /mnt/data/warranty_analysis_output/warranty_guarantee_product_report.csv


In [9]:
# فراوانی اصطلاحات شناسایی‌شده در نظرات فیلترشده
if len(aspect_reviews):
    term_frequency = (
        aspect_reviews["matched_warranty_terms"]
        .str.split("; ")
        .explode()
        .value_counts()
        .rename_axis("matched_term")
        .reset_index(name="review_count")
    )
else:
    term_frequency = pd.DataFrame(columns=["matched_term", "review_count"])

term_frequency.to_csv(OUTPUT_DIR / "warranty_matched_term_frequency.csv", index=False, encoding="utf-8-sig")

audit = pd.DataFrame([{
    "input_file": str(DATA_PATH),
    "total_reviews": len(df),
    "matched_reviews": len(aspect_reviews),
    "matched_share": len(aspect_reviews) / max(len(df), 1),
    "unique_products_with_matches": aspect_reviews["__product_id"].nunique(),
    "rating_column_detected": rating_col if rating_col else "None",
    "mean_overall_calculated": bool(rating_col is not None and product_report["rating_count"].sum() > 0),
    "word2vec_vocabulary_size": len(w2v_model.wv),
    "unique_matching_terms": len(term_frequency),
}])

display(audit)
display(term_frequency.head(30))
audit.to_csv(OUTPUT_DIR / "warranty_analysis_audit.csv", index=False, encoding="utf-8-sig")

print("\nAll output files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")

,input_file,total_reviews,matched_reviews,matched_share,unique_products_with_matches,rating_column_detected,mean_overall_calculated,word2vec_vocabulary_size,unique_matching_terms
0,sentiment_ready_dataset.csv,19997,2401,0.120068,2142,None,False,19285,26


,matched_term,review_count
0,replacement,842
1,warranty,561
2,manufacturer,351
3,protection,337
4,repair,151
5,covered,133
6,protected,110
7,rma,96
8,coverage,77
9,covering,72



All output files:
- warranty_analysis_audit.csv (265 bytes)
- warranty_embedding_candidates.csv (1,496 bytes)
- warranty_fuzzy_candidates.csv (138 bytes)
- warranty_guarantee_matched_reviews.csv (2,842,684 bytes)
- warranty_guarantee_product_report.csv (366,932 bytes)
- warranty_matched_term_frequency.csv (413 bytes)
